# US Accidents - Etapa 3 y 4

Dataset: `US_Accidents_March23.csv`

In [1]:
import os, sys, time
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master('local[*]')
         .appName('US-Accidents-Etapa-3-4')
         .config('spark.ui.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)

c:\Users\oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0


In [2]:
df = spark.read.csv('US_Accidents_March23.csv', header=True, inferSchema=True)
print('Filas:', df.count(), '| Columnas:', len(df.columns))
df.printSchema()

Filas: 7728394 | Columnas: 46
root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): d

## ETAPA 3 - Agregaciones y tratamiento avanzado

In [3]:
from pyspark.sql.functions import col, sum as fsum, avg, count, desc, month, year, to_timestamp

# Agregación por Estado: cantidad de accidentes, severidad promedio y distancia total afectada
agg_estado = (df.groupBy("State")
                .agg(count("*").alias("total_accidentes"),
                     avg("Severity").alias("severidad_promedio"),
                     fsum("Distance(mi)").alias("distancia_total_mi"))
                .orderBy(desc("total_accidentes")))
agg_estado.show(10)

+-----+----------------+------------------+------------------+
|State|total_accidentes|severidad_promedio|distancia_total_mi|
+-----+----------------+------------------+------------------+
|   CA|         1741433|2.1656876836490406|  843238.965983731|
|   FL|          880192|2.1400603504689886| 468051.5110172154|
|   TX|          582837|2.2241244121426744|164855.31801283886|
|   SC|          382557|2.1110553460007266|116980.60296563923|
|   NY|          347960| 2.259549948269916|261513.55793050944|
|   NC|          338199| 2.133823577242984|101442.47499933455|
|   VA|          303301|2.2789736928002218|231132.69705145105|
|   PA|          296620| 2.205764951790169| 270151.4620437136|
|   MN|          192084|  2.16201765894088| 157663.5840487794|
|   OR|          179660| 2.112406768340198|184143.01098319152|
+-----+----------------+------------------+------------------+
only showing top 10 rows


**Análisis:** `groupBy("State") + agg(count, avg, sum)` resume ~7.7M de filas en una tabla de ~49 estados, permitiendo comparar volumen de accidentes, severidad promedio y distancia afectada por estado en una sola pasada sobre los datos.

In [4]:
# Extraer el mes desde Start_Time (formato yyyy-MM-dd HH:mm:ss)
dfm = df.withColumn("Mes", month(to_timestamp(col("Start_Time"))))
dfm.select("Start_Time", "Mes", "Severity").show(5)

+-------------------+---+--------+
|         Start_Time|Mes|Severity|
+-------------------+---+--------+
|2016-02-08 05:46:00|  2|       3|
|2016-02-08 06:07:59|  2|       2|
|2016-02-08 06:49:27|  2|       2|
|2016-02-08 07:23:34|  2|       3|
|2016-02-08 07:39:07|  2|       2|
+-------------------+---+--------+
only showing top 5 rows


In [5]:
# Agregación mensual: cantidad de accidentes y severidad promedio por mes
agg_mensual = (dfm.groupBy("Mes")
                  .agg(count("*").alias("total_accidentes"),
                       avg("Severity").alias("severidad_promedio"))
                  .orderBy("Mes"))
agg_mensual.show(12)

+---+----------------+------------------+
|Mes|total_accidentes|severidad_promedio|
+---+----------------+------------------+
|  1|          751946|2.1913342181486435|
|  2|          658984| 2.204041372779916|
|  3|          554595|2.2326093816208226|
|  4|          587300| 2.215571258300698|
|  5|          558176|2.2208156567104282|
|  6|          571373| 2.239978087869045|
|  7|          512335|  2.24311046483258|
|  8|          599666| 2.236888534617604|
|  9|          651381| 2.214799019314349|
| 10|          675130|2.2141365366670125|
| 11|          760165| 2.192929166694073|
| 12|          847343| 2.176228516669165|
+---+----------------+------------------+



**Análisis:** Extraer el mes con `month(to_timestamp(...))` permite ver estacionalidad: se espera que los meses de invierno (nov-feb) concentren más accidentes por condiciones climáticas adversas (nieve, hielo, baja visibilidad).

In [6]:
# Revisión de duplicados
total = df.count()
unicos = df.dropDuplicates().count()
print("Filas totales:", total, "| Sin duplicados:", unicos, "| Duplicados:", total - unicos)

# Duplicados considerando solo columnas clave (mismo punto, mismo instante)
unicos_clave = df.dropDuplicates(["Start_Time", "Start_Lat", "Start_Lng"]).count()
print("Sin duplicados por (Start_Time, Start_Lat, Start_Lng):", unicos_clave)

Filas totales: 7728394 | Sin duplicados: 7728394 | Duplicados: 0
Sin duplicados por (Start_Time, Start_Lat, Start_Lng): 7186791


**Análisis:** `dropDuplicates()` sobre todas las columnas detecta registros exactamente repetidos (por ejemplo, cargas duplicadas del CSV). Usar una clave más flexible (`Start_Time`, `Start_Lat`, `Start_Lng`) ayuda a detectar posibles duplicados lógicos (mismo accidente reportado dos veces) que no son idénticos en todas las columnas.

In [7]:
# Detección de outliers con approxQuantile sobre Distance(mi)
q1, q2, q3 = df.approxQuantile("Distance(mi)", [0.25, 0.50, 0.75], 0.01)
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
print("Q1:", round(q1, 2), "| Q2 (mediana):", round(q2, 2), "| Q3:", round(q3, 2), "| Límite superior:", round(lim_sup, 2))

outliers = df.filter(col("Distance(mi)") > lim_sup).count()
print("Accidentes con distancia atípica (> límite superior):", outliers)

Q1: 0.0 | Q2 (mediana): 0.03 | Q3: 0.46 | Límite superior: 1.15
Accidentes con distancia atípica (> límite superior): 969428


**Análisis:** `approxQuantile` evita el costo de un ordenamiento exacto sobre millones de filas (usa un algoritmo aproximado con tolerancia 0.01). El rango intercuartílico (IQR) permite fijar un límite superior razonable: los accidentes con `Distance(mi)` muy por encima de ese límite probablemente correspondan a cierres de vía extensos o errores de medición, más que a un accidente puntual.

In [8]:
# Comparación de tiempos con y sin cache() al reutilizar el mismo DataFrame varias veces
df_severo = df.filter(col("Severity") >= 3)

t0 = time.time()
df_severo.count()
df_severo.groupBy("State").count().collect()
df_severo.agg(avg("Distance(mi)")).collect()
t1 = time.time()
print("Sin cache: {:.2f} s".format(t1 - t0))

df_severo.cache()
df_severo.count()  # materializa la cache

t2 = time.time()
df_severo.count()
df_severo.groupBy("State").count().collect()
df_severo.agg(avg("Distance(mi)")).collect()
t3 = time.time()
print("Con cache: {:.2f} s".format(t3 - t2))

df_severo.unpersist()

Sin cache: 11.21 s
Con cache: 1.00 s


DataFrame[ID: string, Source: string, Severity: int, Start_Time: timestamp, End_Time: timestamp, Start_Lat: double, Start_Lng: double, End_Lat: double, End_Lng: double, Distance(mi): double, Description: string, Street: string, City: string, County: string, State: string, Zipcode: string, Country: string, Timezone: string, Airport_Code: string, Weather_Timestamp: timestamp, Temperature(F): double, Wind_Chill(F): double, Humidity(%): double, Pressure(in): double, Visibility(mi): double, Wind_Direction: string, Wind_Speed(mph): double, Precipitation(in): double, Weather_Condition: string, Amenity: boolean, Bump: boolean, Crossing: boolean, Give_Way: boolean, Junction: boolean, No_Exit: boolean, Railway: boolean, Roundabout: boolean, Station: boolean, Stop: boolean, Traffic_Calming: boolean, Traffic_Signal: boolean, Turning_Loop: boolean, Sunrise_Sunset: string, Civil_Twilight: string, Nautical_Twilight: string, Astronomical_Twilight: string]

**Análisis:** Al reutilizar `df_severo` en tres acciones distintas (`count`, `groupBy`, `agg`), sin `cache()` Spark vuelve a leer y filtrar el CSV completo en cada acción. Con `cache()` (materializada con un `count()` inicial), las acciones posteriores reutilizan el DataFrame en memoria, reduciendo notablemente el tiempo total. `unpersist()` libera la memoria una vez que ya no se necesita.

### ANÁLISIS Y HALLAZGOS (escrito por la pareja)

_[Espacio para que la pareja documente sus hallazgos de la Etapa 3: qué estados concentran más accidentes, cómo varía la severidad promedio, en qué meses hay más accidentes, cuántos duplicados y outliers se encontraron, y cuánto mejoró el tiempo de ejecución al usar `cache()`.]_

## ETAPA 4 - Gráficas con Plotly

In [9]:
import plotly.express as px

# Gráfica básica: barras con el Top 10 de estados con más accidentes (sobre el agregado, no el crudo)
top_estados_pd = agg_estado.limit(10).toPandas()

fig_bar = px.bar(top_estados_pd, x="State", y="total_accidentes", text="total_accidentes",
                  title="Top 10 estados con más accidentes",
                  color="State", color_discrete_sequence=px.colors.qualitative.Set2)
fig_bar.update_traces(texttemplate="%{y:,}", textposition="outside")
fig_bar.show()

c:\Users\oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


**Análisis:** Se grafica `agg_estado` (ya agregado en Spark, solo 49 filas) en lugar del DataFrame crudo de millones de registros. La gráfica de barras muestra de forma directa qué estados concentran el mayor volumen de accidentes reportados.

In [12]:
# Gráfica avanzada: heatmap (pivot) de cantidad de accidentes por Mes y Severidad
pivot_mes_sev = (dfm.groupBy("Mes")
                    .pivot("Severity")
                    .agg(count(col("ID")))
                    .orderBy("Mes")
                    .toPandas()
                    .fillna(0)
                    .set_index("Mes"))

fig_heat = px.imshow(pivot_mes_sev.T,
                      labels=dict(x="Mes", y="Severidad", color="N° accidentes"),
                      title="Heatmap: accidentes por Mes y Severidad",
                      color_continuous_scale="YlOrRd")
fig_heat.show()

c:\Users\oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


**Análisis:** El `pivot("Severity")` se calcula dentro de Spark (agregando sobre todo el dataset) y solo la tabla resultante (meses x niveles de severidad) se trae a Pandas con `toPandas()`. El heatmap permite identificar rápidamente combinaciones mes/severidad con mayor concentración de accidentes, algo difícil de leer en una tabla numérica.

### ANÁLISIS Y HALLAZGOS (escrito por la pareja)

_[Espacio para que la pareja documente sus hallazgos de la Etapa 4: qué estados destacan en la gráfica de barras, qué patrones mes/severidad revela el heatmap, y qué decisiones de diseño (agregar antes de graficar) tomaron para evitar graficar el dataset crudo.]_

In [13]:
spark.stop()
print("Sesión cerrada")

Sesión cerrada
